In [1]:
import sqlite3
import pandas as pd
import numpy as np

# DB 생성 및 연결
conn = sqlite3.connect('loan_data.db')
cursor = conn.cursor()
print("DB 연결 완료!")

DB 연결 완료!


In [2]:
cursor.executescript('''
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS loans;

CREATE TABLE customers (
    customer_id     INTEGER PRIMARY KEY,
    age             INTEGER,
    income          REAL,
    home_ownership  TEXT,
    emp_length      REAL
);

CREATE TABLE loans (
    loan_id         INTEGER PRIMARY KEY,
    customer_id     INTEGER,
    loan_amount     REAL,
    loan_intent     TEXT,
    loan_grade      TEXT,
    int_rate        REAL,
    loan_status     INTEGER,
    percent_income  REAL,
    default_history TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
''')
conn.commit()
print("테이블 생성 완료!")

테이블 생성 완료!


In [3]:
df = pd.read_csv('../Loan-Default-Prediction/data/credit_risk_dataset.csv')
df = df.dropna()
df = df[df['person_age'] < 100]
df = df[df['person_emp_length'] < 60]
df = df.reset_index(drop=True)
df['customer_id'] = df.index + 1
df['loan_id'] = df.index + 1

# customers 테이블
customers = df[['customer_id', 'person_age', 'person_income',
                'person_home_ownership', 'person_emp_length']]
customers.columns = ['customer_id', 'age', 'income', 'home_ownership', 'emp_length']
customers.to_sql('customers', conn, if_exists='replace', index=False)

# loans 테이블
loans = df[['loan_id', 'customer_id', 'loan_amnt', 'loan_intent',
            'loan_grade', 'loan_int_rate', 'loan_status',
            'loan_percent_income', 'cb_person_default_on_file']]
loans.columns = ['loan_id', 'customer_id', 'loan_amount', 'loan_intent',
                 'loan_grade', 'int_rate', 'loan_status',
                 'percent_income', 'default_history']
loans.to_sql('loans', conn, if_exists='replace', index=False)

print(f"customers 테이블: {pd.read_sql('SELECT COUNT(*) FROM customers', conn).iloc[0,0]}건")
print(f"loans 테이블: {pd.read_sql('SELECT COUNT(*) FROM loans', conn).iloc[0,0]}건")

customers 테이블: 28632건
loans 테이블: 28632건


In [4]:
print("=== customers 샘플 ===")
display(pd.read_sql("SELECT * FROM customers LIMIT 5", conn))

print("=== loans 샘플 ===")
display(pd.read_sql("SELECT * FROM loans LIMIT 5", conn))

=== customers 샘플 ===


,customer_id,age,income,home_ownership,emp_length
0,1,21,9600,OWN,5.0
1,2,25,9600,MORTGAGE,1.0
2,3,23,65500,RENT,4.0
3,4,24,54400,RENT,8.0
4,5,21,9900,OWN,2.0


=== loans 샘플 ===


,loan_id,customer_id,loan_amount,loan_intent,loan_grade,int_rate,loan_status,percent_income,default_history
0,1,1,1000,EDUCATION,B,11.14,0,0.10,N
1,2,2,5500,MEDICAL,C,12.87,1,0.57,N
2,3,3,35000,MEDICAL,C,15.23,1,0.53,N
3,4,4,35000,MEDICAL,C,14.27,1,0.55,Y
4,5,5,2500,VENTURE,A,7.14,1,0.25,N
